# 2. Feature-Extraktion: Loop-Features + HRV

Setzt bei `df_analysis` + `df_annotations` aus [`01_load_filter_annotate_visualize.ipynb`](01_load_filter_annotate_visualize.ipynb) an (hier zur Eigenständigkeit kurz neu berechnet) und zeigt:

1. Beat-zu-Beat-Rotation (`df_r`) — Grundlage für Loop-Features und HRV
2. P-/QRS-/T-Loop-Geometrie-Features (Fläche, Rundheit, Asymmetrie, Winkel, ...)
3. HRV-Kennwerte (SDNN, RMSSD, PNS-/SNS-Index, ...)
4. ECG-derived Respiration (EDR) als Bonus

Auch dieses Notebook dient als Funktionskontrolle: Wenn `df_full` am Ende plausibel viele Beats × Features enthält und die HRV-Werte physiologisch sinnvoll sind, funktioniert die Feature-Pipeline korrekt.

In [1]:
import vcgsuite as ecg
from vcgsuite.features.loop.p_wave import compute_p_wave_features
from vcgsuite.features.loop.t_wave import build_vagus_features, compute_t_wave_features
from vcgsuite.features.loop.qrs_complex import compute_qrs_features
from vcgsuite.features.loop.merge import merge_loop_features
from vcgsuite.viz.loop_features import plot_p_loop_features, plot_qrs_loop_features, plot_t_loop_features
from vcgsuite.hrv.prep import build_rr_dataframe
from vcgsuite.hrv.helpers import compute_hrv_full
from vcgsuite.hrv.analysis import extract_edr_signal
from vcgsuite.viz.hrv_dashboard import plot_kubios_ans_balance
import plotly.graph_objects as go

## 0. Ausgangsdaten (siehe Notebook 1 für Details)

In [2]:
DATA_PATH = "../../../sample_data/-I-2025-4-6_6min_2brust_2bauchOhne_2bauchmit.txt"
MODE = "easi"
DURATION_S = 90.0

df_analysis = ecg.load_and_process(DATA_PATH, mode=MODE, duration=DURATION_S)
df_analysis, dt = ecg.compute_vcg_kinematics(df_analysis)
df_analysis.attrs["fs"] = df_analysis.attrs.get("fs", 1.0 / dt)

r_peak_times, _ = ecg.detect_r_peaks(df_analysis)
r_turn_times, _ = ecg.detect_r_turn(df_analysis, r_peak_times)
df_annotations = ecg.annotate_all_beats(df_analysis, r_peak_times, r_turn_times)

print(f"{len(df_annotations)} Beats annotiert.")


  Pipeline : EASI  |  -I-2025-4-6_6min_2brust_2bauchOhne_2bauchmit.txt
  Proband  : 2025  →  S2025
EASI  |  Proband: 2025  |  Fenster: 0.0 s  →  90.0 s  (22500 Samples = 90.0 s)
       Spalten: ['R', 'M', 'L']
  [Filter] ZapLine 50.0 Hz  →  FIR HP 1.5 Hz / LP 37.5 Hz
Power of components removed by DSS: 0.08
ZapLine: 50.0 Hz, bis 2. Harmonische, nremove=1
FIR: HP 1.5 Hz (201 Taps)  →  LP 37.5 Hz (21 Taps)
  [VCG]    EASI → Frank XYZ  (W · T)

  df_analysis: (22500, 7)  |  Proband: S2025

Sampling-Rate:     250.0 Hz
Fensterbreite:     10.0 s  (2500 Samples)
Detektierte Peaks: 101
Ø RR-Abstand:      893.1 ms
Ø HF:              67.2 bpm
RR-Bereich:        680 – 1128 ms
R_turn gefunden:   101 / 101
Ø Δ R_peak→R_turn: 12.0 ms  (erwartet ~12 ms)
Min/Max Δ:         12.0 / 12.0 ms
Annotierte Beats: 101

Marker        gefunden   Ø rel. Zeit
----------------------------------------
  R_turn            101       +12.0 ms
  S_on              101       +29.1 ms
  Q_on              100       -60.8 m

## 1. Beat-zu-Beat-Rotation

`df_r`: pro Beat der Zeigerbetrag `r` und die komplexe Amplitude `|phi + i*theta|` an R_peak und R_turn, sowie das RR-Intervall — Grundlage für alle nachfolgenden Feature- und HRV-Berechnungen.

In [3]:
df_r = ecg.compute_beat_rotation(df_annotations, df_analysis)
df_r.head()

,beat_id,t_R_peak+,Rpeak_r,Rpeak_ca,Rturn_r,Rturn_ca,RR_ms
0,0,0.012,0.507589,1.036906,0.360539,2.168624,NaN
1,1,0.772,0.701450,0.661100,0.502551,0.680666,760.0
2,2,1.584,0.634707,0.804203,0.577716,0.462341,812.0
3,3,2.460,0.598804,0.833093,0.582669,0.434681,876.0
4,4,3.344,0.703709,1.014290,0.577875,0.274408,884.0


## 2. P-/QRS-/T-Loop-Features

Jede Welle wird als 3D-Loop-Segment betrachtet (Fläche, SVD-Rundheit/Planarität, Asymmetrie, Winkel zu benachbarten Wellen). Reihenfolge ist wichtig: `df_p` und `df_vagus` werden zuerst gebraucht (siehe Kommentare in `vcgsuite/features/loop/t_wave.py`).

In [4]:
df_p = compute_p_wave_features(df_analysis, df_annotations, df_r)
df_vagus = build_vagus_features(df_analysis, df_annotations)
df_qrs = compute_qrs_features(df_analysis, df_annotations, df_vagus, df_p, df_r)
df_t = compute_t_wave_features(df_analysis, df_annotations, df_r, df_p, df_vagus=df_vagus)

df_full = merge_loop_features(df_p, df_qrs, df_t, df_r)
df_full.head()

Beats: 101  |  VCG-Samples: 22500  |  fs=250.0 Hz
       P_dur_ms     PQ_ms  P_rise_ms  P_symmetry    P_area     P_rho  \
count  100.0000  100.0000   100.0000    100.0000  100.0000  100.0000   
mean   120.6200  121.9400    49.7800      0.4190    0.0073    0.1206   
std     21.7659   17.1257    11.7272      0.1011    0.0030    0.0547   
min     44.0000   78.0000    24.0000      0.2857    0.0023    0.0190   
25%    109.0000  112.0000    41.5000      0.3520    0.0054    0.0925   
50%    126.0000  122.0000    48.0000      0.4000    0.0068    0.1151   
75%    134.5000  132.0000    58.0000      0.4676    0.0086    0.1408   
max    168.0000  164.0000    76.0000      0.8387    0.0262    0.4157   

          P_phi    P_asym  P_dipol_norm   P_round  theta_P_QRS  
count  100.0000  100.0000      100.0000  100.0000     100.0000  
mean     0.0087    0.3177        0.0065    0.4363      97.3732  
std      0.0095    0.3455        0.0011    0.1472      14.7164  
min      0.0005   -0.3801        0.0028  

,beat_id,theta_P_QRS,P_asym,P_phi,P_rho,P_dipol_norm,P_area,P_dur_ms,PQ_ms,QRS_area,...,T_asym,T_width_ms,QTc_Bazett,T_rise_ms,T_fall_ms,RR_ms,resp_phase,t_R_peak,Rpeak_ca,Rpeak_r
0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.410033,250.0,NaN,80.0,170.0,NaN,NaN,0.012,1.036906,0.507589
1,1,102.237645,0.533393,0.006960,0.168193,0.007605,0.008301,168.0,160.0,0.214967,...,1.369932,292.0,594.186360,82.0,210.0,760.0,NaN,0.772,0.661100,0.701450
2,2,112.086862,0.028379,0.032636,0.176140,0.007267,0.011840,140.0,138.0,0.228075,...,0.730345,302.0,608.138189,82.0,220.0,812.0,NaN,1.584,0.804203,0.634707
3,3,109.317820,0.213762,0.009120,0.119463,0.006509,0.008049,118.0,110.0,0.206724,...,1.628284,338.0,594.049303,106.0,232.0,876.0,NaN,2.460,0.833093,0.598804
4,4,123.537399,0.560388,0.017877,0.109932,0.005412,0.009766,92.0,108.0,0.373040,...,1.235611,264.0,544.557298,82.0,182.0,884.0,NaN,3.344,1.014290,0.703709


In [5]:
# Kontrolle: Anteil fehlender Werte pro Feature (viele NaN deuten auf Annotationslücken hin)
feat_cols = [c for c in df_full.columns if c != "beat_id"]
missing_pct = (df_full[feat_cols].isna().mean() * 100).round(1).sort_values(ascending=False)
missing_pct.head(10)

resp_phase      100.0
theta_P_QRS       1.0
G_QRS             1.0
theta_QT_deg      1.0
theta_P_T         1.0
RR_ms             1.0
P_asym            1.0
QRS_sym           1.0
QRS_dur_ms        1.0
theta_QRS_P       1.0
dtype: float64

In [6]:
plot_qrs_loop_features(df_qrs).show()

In [7]:
plot_p_loop_features(df_p).show()

In [8]:
plot_t_loop_features(df_t).show()

**Kontrolle:** Die geglätteten Linien sollten über die 90 s einigermaßen stabil um einen physiologisch plausiblen Wert verlaufen, ohne abrupte Sprünge auf Null oder Ausreißer (Ausreißer deuten meist auf einzelne fehlgeschlagene Annotationen hin, siehe `missing_pct` oben).

## 3. HRV-Kennwerte

In [9]:
df_rr = build_rr_dataframe(df_r)
hrv = compute_hrv_full(df_rr["RR_interval"].to_numpy())

for key in ("mean_hr", "sdnn", "rmssd", "pnn50", "SD1", "SD2", "SI", "PNS_index", "SNS_index"):
    print(f"{key:12s} = {hrv[key]:.3f}")

df_rr bereit
   Beats:       101
   Dauer:       89.3 s  (1.5 min)
   ∅ RR:        892 ms  (∅ HR = 67 bpm)
   ∅ Rpeak_ca:  0.7778  (NaN: 0)
   ∅ Rpeak_r:   0.6569  (NaN: 0)
   t-Bereich:   [0.01, 89.32] s
mean_hr      = 67.280
sdnn         = 0.133
rmssd        = 0.080
pnn50        = 42.000
SD1          = 0.057
SD2          = 0.179
SI           = 2.673
PNS_index    = 0.132
SNS_index    = -1.033


In [10]:
plot_kubios_ans_balance(hrv).show()

## 4. Bonus: ECG-derived Respiration (EDR)

Extrahiert ein Atemsurrogat aus der mechanischen Herzrotation (`Rpeak_ca`, Standardkanal). Für ein vollständiges interaktives Dashboard (inkl. CWT-Scalogram, Poincaré-Plot) siehe `vcgsuite.viz.hrv_dashboard.build_edr_dashboard` — die benötigt zusätzlich vorab berechnete CWT-/PSD-Werte (siehe Docstring), hier zeigen wir nur die EDR-Extraktion selbst.

In [11]:
t_edr, edr, f_resp, rpm, (f_w, p_w) = extract_edr_signal(df_rr)

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_edr, y=edr, mode="lines", name="EDR (Rpeak_ca)"))
fig.update_layout(
    template="plotly_dark",
    title=f"ECG-derived Respiration — {rpm:.1f} Atemzüge/min" if rpm else "ECG-derived Respiration",
    xaxis_title="Zeit (s)", yaxis_title="EDR (a.u.)", height=300,
)
fig.show()

---

Aktivierungskarten (Herzmesh + Body-Surface-Potential) aus denselben VCG-Daten folgen in einer späteren Version.